# Semi-implicit ABBA exact-Jacobian symplecticity study

This experiment evaluates `SemiImplicitABBA`, the tangent-propagating form of the second-order endpoint-time ABBA map closed by Hairer's symmetric projection. The physical trajectory is the same as for `SymmetricProjectedABBA`, but every converged main step also evaluates the exact ideal-root physical Jacobian derived in `docs/tex/ABBA_semiimplicit/ABBA_semiimplicit_English.tex`:

$$J_n=D\Psi_{h,t_n}(z_n)=\mathcal P-\mathcal Q\mathcal K^{-1}\mathcal L.$$

The implementation applies this formula with a linear solve, never by forming $\mathcal K^{-1}$. It propagates the complete tangent from the initial condition as

$$M_{n+1}=J_nM_n,\qquad M_0=I.$$

The recorded local and accumulated defects are

$$\varepsilon_{\mathrm{local},n}=\frac{\|J_n^T\Omega J_n-\Omega\|_F}{\|\Omega\|_F},\qquad \varepsilon_{\mathrm{flow},n}=\frac{\|M_n^T\Omega M_n-\Omega\|_F}{\|\Omega\|_F}.$$

No centered-difference Jacobian is evaluated in this study. The transported polygon area, $|\det(M_n)-1|$, Newton residuals, iteration counts, and projection multipliers provide complementary geometric and nonlinear-solver diagnostics.

API migration: this notebook uses the current simulation API. Physical rho and eta belong to dynamics or study settings; initial configurations store geometry. Stored outputs were cleared and should be regenerated before scientific interpretation.


In [ ]:
import numpy as np

from studies import (
    RandomPotentialConfig,
    ImplicitABBASymplecticityConfig,
    centered_circle,
    pi_area_steps,
    run_abba2_reduced_multiplier_symplecticity_study,
)
from visualization import display_animation

## Reproducible configuration

The random-potential seed and spectrum, interpolation order, boundary geometry, guiding-center radius, integration interval, three ABBA steps, observation interval, and Newton stopping parameters are explicit below. Fifth-order interpolation supplies the potential Hessians required by the exact stage Jacobians. There is deliberately no finite-difference scale.

In [ ]:
potential_config = RandomPotentialConfig(
    amplitude=0.7,
    max_wave_number=25,
    nx=64,
    ny=64,
    seed=27,
    interpolation_order=5,
)
potential = potential_config.build()

circle_radius = 0.5
circle_points = 16
rho = 0.3
circle = centered_circle(
    potential,
    radius=circle_radius,
    points=circle_points,
    
)

study_config = ImplicitABBASymplecticityConfig(
    rho=rho,
    steps=pi_area_steps(40, 80, 160),
    t_span=(0.0, 4 * np.pi),
    save_interval=np.pi / 8,
    chunk_size=16,
    progress=True,
    block_prefix="semiimplicit_abba_exact_symplecticity",
    newton_absolute_tolerance=1e-13,
    newton_relative_tolerance=1e-12,
    newton_max_iterations=12,
)

print(potential_config)
print(study_config)
print(
    f"Circle: {circle_points} points; t={study_config.t_span}; "
    f"{study_config.output_sample_count} saved states; "
    "step Jacobian=exact implicit-function tangent"
)

## Semi-implicit ABBA integrations and persisted exact Jacobians

Newton starts from $\mu_0=0$ on every step, uses the infinity norm for its stopping test, and reevaluates the exact residual Jacobian after each correction. At convergence, the method evaluates the exact physical tangent $J_n$, propagates $M_n$, and gives the same local matrix to the observer. Scalar diagnostics and full local and accumulated matrices are persisted at the common observation times.

In [ ]:
result = run_abba2_reduced_multiplier_symplecticity_study(
    potential,
    circle,
    notebook_path=(
        "notebooks/experiments/symplecticity/"
        "gc_area_and_semiimplicit_abba_symplecticity.ipynb"
    ),
    config=study_config,
    jacobian_method="implicit_function",
    metadata={
        **potential_config.metadata(),
        "circle_radius": circle_radius,
        "study_kind": "versioned_exact_jacobian_symplecticity_experiment",
    },
)
result.print_summary()

## Exact-tangent audit

Each solution must identify its tangent as the implicit-function Jacobian and retain the final accumulated matrix. These checks ensure that the experiment did not silently fall back to numerical differentiation.

In [ ]:
assert result.jacobian_method == "implicit_function"
for step in study_config.steps:
    solution = result.solutions[step.label]
    assert solution.diagnostics["projection_formulation"] == "reduced_multiplier"
    records = result.records[step.label]
    assert records
    assert np.isclose(records[-1].time, solution.t[-1])
    final = records[-1]
    assert np.isfinite(final.condition_number)
    assert np.isfinite(final.relative_defect)
    print(
        f"{step.label}: exact implicit-function tangent; "
        f"condition number={final.condition_number:.8e}, "
        f"relative symplecticity defect={final.relative_defect:.8e}"
    )


## Time-dependent diagnostics

The local matrix defect tests each projected ABBA tangent independently. The accumulated defect and determinant error test the complete discrete flow from the initial boundary, while the area panel shows the transported polygon geometry.

In [ ]:
diagnostic_figure, diagnostic_axes = result.plot_diagnostics()

## Exact-Jacobian symplecticity floor

For an exact root, the physical ABBA tangent is structurally symplectic. Here its defect is evaluated directly from the implemented Jacobian, so there is no finite-difference truncation or cancellation error. The remaining floor reflects floating-point arithmetic, potential-Hessian evaluation, the finite Newton stopping tolerance, and accumulation over many steps; it is not a temporal convergence error.

In [ ]:
defect_floor_figure, defect_floor_axis = result.plot_defect_floor()

## Nonlinear projection diagnostics

The iteration and residual histories verify that the projection solve remains below its configured stopping threshold. The maximum multiplier is expected to decrease approximately as $O(h^3)$; this is a consistency property of the symmetric projection, not the global trajectory order.

In [ ]:
solver_figure, solver_axes = result.plot_solver_diagnostics()

## Comparative animation

The animation synchronizes the effective potential, transported ABBA contours, relative polygon-area error, and accumulated exact-tangent symplecticity defect. The same color identifies each step size in every panel.

In [ ]:
display_animation(
    result.animate(
        frames=None,
        interval=120,
    )
)

## Interpretation

This experiment measures the implemented ideal-root tangent rather than a centered-difference approximation of the finite-iteration control flow. Local defects test the exact formula $\mathcal P-\mathcal Q\operatorname{solve}(\mathcal K,\mathcal L)$ at each converged stage set; accumulated defects test the ordered product of those matrices. Small nonzero values are therefore a numerical floor from Newton tolerance, interpolated Hessians, and floating-point accumulation. Polygonal area error remains a separate finite-boundary diagnostic and should not be identified with the matrix symplecticity defect.